
# Learning and inference of latents from striatal recordings while a mouse performed a learned poking sequence

The code below learns and infers latents from strialt recording while a mouse
performed a learned poking sequence. Recordings from [Thompson et al.,
2026](https://www.nature.com/articles/s41593-026-02362-5).


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joacorapela/lds_itba_2026/blob/main/code/notebooks/solutions/plotStriatum.ipynb)

In [9]:
!pip install git+https://github.com/joacorapela/gcnu_common.git@numpy
!pip install git+https://github.com/joacorapela/ssm.git@master

  Cloning https://github.com/joacorapela/gcnu_common.git (to revision numpy) to /tmp/pip-req-build-cyy2kn0c
  Running command git clone --filter=blob:none --quiet https://github.com/joacorapela/gcnu_common.git /tmp/pip-req-build-cyy2kn0c
  Running command git checkout -b numpy --track origin/numpy
  Switched to a new branch 'numpy'
  branch 'numpy' set up to track 'origin/numpy'.
  Resolved https://github.com/joacorapela/gcnu_common.git to commit 2d56d107adfc51239deff7d6ef17c850fb0132f5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gcnu_common: filename=gcnu_common-0.0.15-py3-none-any.whl size=14473 sha256=602a36dfe5f78764ef5f227bb086d78ebcc72a0728d9d11e7dc344a32febea69
  Stored in directory: /tmp/pip-ephem-wheel-cache-5de7va4y/wheels/6a/11/d4/7a122bf391590037cb03c1ca56352bb9347afb8e497a6374d4
Successfully built gcnu_common
  Cloning https://github.com/joacorapela/ssm.git (to re

In [10]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import gcnu_common.utils.neural_data_analysis
import ssm.inference
import ssm.learning
import ssm.plotting
import ssm.neural_latents.utils
import ssm.neural_latents.plotting

## Define variables



In [11]:
# data
bin_size = 0.2  # 200~ms
sqrt_transform = True
units_info_filename = "https://www.gatsby.ucl.ac.uk/~rapela/emmett/data/good_units_df.csv"
transition_data_filename = "https://www.gatsby.ucl.ac.uk/~rapela/emmett/data/Transition_data_sync.csv"
spikes_times_col_name = "Spike_times"

## for EJTxxx_implantx/recordingx_xx-xx-xxxx clusters 0-266 belong to the striatum
clusters_indices = np.arange(266)

# EM
n_latents = 5
n_clusters = clusters_indices.size
EM_start_time_sec = 8400.0
EM_duration_sec = 300.0
EM_max_iter = 5000
EM_tol = 1e-1
min_spike_rate_var = 0.01 # minimal spike rate variance
EM_constraint_diag_R = True
EM_min_spike_rate_var = 0.01

vars_to_estimate = {
    "u": False,
    "B": True,
    "Q": True,
    "a": True,
    "Z": True,
    "R": True,
    "m0": True,
    "V0": True,
}

# EM initial conditions
constant_u = 0.0
mean_B = 0.1
sigma_B = 0.3
mean_Q = 5.0
sigma_Q = 0.5
mean_a = 1.0
sigma_a = 0.1
mean_Z = 0.01
sigma_Z = 0.005
mean_R = 1.0
sigma_R = 0.2
constant_m0 = 0.0
mean_V0 = 0.02
sigma_V0 = 0.005

# filtering
filtering_start_time_sec = 8400.0
filtering_duration_sec = 300.0

# plotting
plotting_start_time_sec = 8400.0
plotting_duration_sec = 300.0
skip_orthonormalization = False
ports_to_plot = [1,2,3,4,5,6,7]
ports_linetypes_list = ["solid", "solid", "solid", "solid", "solid", "solid", "solid"]
ports_colors_list = ["green", "red", "cyan", "yellow", "purple", "blue", "magenta"]

ports_linetypes = dict(zip(ports_to_plot, ports_linetypes_list))
ports_colors = dict(zip(ports_to_plot, ports_colors_list))

u0 = constant_u * np.ones(shape=(n_latents, 1))
B0 = np.random.normal(loc=mean_B, scale=sigma_B, size=(n_latents, n_latents))
a0 = np.random.normal(loc=mean_a, scale=sigma_a, size=(n_clusters, 1))
Z0 = np.random.normal(loc=mean_Z, scale=sigma_Z, size=(n_clusters, n_latents))
Q0 = np.diag(np.abs(np.random.normal(loc=mean_Q, scale=sigma_Q,
                                     size=n_latents)))
R0 = np.diag(np.abs(np.random.normal(loc=mean_R, scale=sigma_R,
                                     size=n_clusters)))
m0_0 = constant_m0 * np.ones(shape=n_latents)
V0_0 = np.diag(np.abs(np.random.normal(loc=mean_V0, scale=sigma_V0,
                                       size=n_latents)))

## Bin spikes times



In [13]:
# read data
units_info_df = pd.read_csv(units_info_filename)

# get spikes timestamps
spikes_timestamps = [None for n in range(n_clusters)]
for cluster_index in clusters_indices:
    unit_spikes_times_str = units_info_df.iloc[cluster_index][spikes_times_col_name][1:-1].split(",")
    unit_spikes_times = np.array([float(unit_spike_times_str) for unit_spike_times_str in unit_spikes_times_str])
    spikes_timestamps[cluster_index] = unit_spikes_times

# bin spikes
t_min = min(min(st) for st in spikes_timestamps if len(st) > 0)
t_max = max(max(st) for st in spikes_timestamps if len(st) > 0)
bins_edges = np.arange(t_min, t_max, bin_size)
bins_centers = (bins_edges[1:] + bins_edges[:-1])/2

binned_spikes = np.empty(shape=(n_clusters, len(bins_centers)))
for n in range(n_clusters):
    binned_spikes[n, :] = gcnu_common.utils.neural_data_analysis.binSpikesTimes(
        spikes_times=spikes_timestamps[n], bins_edges=bins_edges,
        time_unit="sec")

if sqrt_transform:
    binned_spikes = np.sqrt(binned_spikes)

## Perform EM



In [14]:
EM_data = binned_spikes.T
EM_end_time_sec = EM_start_time_sec + EM_duration_sec
valid_bins_mask = np.logical_and(EM_start_time_sec <= bins_centers,
                                 bins_centers < EM_end_time_sec)
EM_data = EM_data[valid_bins_mask, :]

spike_rate_vars = np.var(EM_data, axis=0)
valid_clusters_mask = spike_rate_vars > min_spike_rate_var
EM_data = EM_data[:, valid_clusters_mask]
valid_clusters_indices = clusters_indices[valid_clusters_mask]

a0 = a0[valid_clusters_mask]
Z0 = Z0[valid_clusters_mask,:]
R0 = R0[valid_clusters_mask, :][:, valid_clusters_mask]

# make sure that the first EM_data point is not NaN
first_not_nan_index = np.where(~np.isnan(EM_data).any(axis=1))[0][0]
EM_data = EM_data[first_not_nan_index:,]

optim_res = ssm.learning.em_SS_LDS(
    y=EM_data.T, u0=u0, B0=B0, Q0=Q0, a0=a0, Z0=Z0, R0=R0,
    m0_0=m0_0, V0_0=V0_0, max_iter=EM_max_iter, tol=EM_tol,
    vars_to_estimate=vars_to_estimate,
    constraint_diag_R=EM_constraint_diag_R,
)

LogLike[0000]=-649944.905613
LogLike[0001]=-284821.405143
LogLike[0002]=-276532.670167
LogLike[0003]=-272154.797916
LogLike[0004]=-270054.543541
LogLike[0005]=-268873.586984
LogLike[0006]=-268240.056599
LogLike[0007]=-267926.903691
LogLike[0008]=-267783.387292
LogLike[0009]=-267716.804185
LogLike[0010]=-267680.785795
LogLike[0011]=-267656.012536
LogLike[0012]=-267635.475587
LogLike[0013]=-267616.913248
LogLike[0014]=-267599.660636
LogLike[0015]=-267583.516748
LogLike[0016]=-267568.387653
LogLike[0017]=-267554.192844
LogLike[0018]=-267540.849800
LogLike[0019]=-267528.276904
LogLike[0020]=-267516.397894
LogLike[0021]=-267505.144386
LogLike[0022]=-267494.456582
LogLike[0023]=-267484.282877
LogLike[0024]=-267474.578954
LogLike[0025]=-267465.306732
LogLike[0026]=-267456.433344
LogLike[0027]=-267447.930239
LogLike[0028]=-267439.772418
LogLike[0029]=-267431.937813
LogLike[0030]=-267424.406779
LogLike[0031]=-267417.161692
LogLike[0032]=-267410.186621
LogLike[0033]=-267403.467072
LogLike[0034]=

## Perform filtering



In [15]:
filtering_data = binned_spikes.T
filtering_data = filtering_data[:, valid_clusters_mask]
filtering_end_time_sec = filtering_start_time_sec + filtering_duration_sec
filtering_valid_bins_mask = np.logical_and(
    filtering_start_time_sec <= bins_centers,
    bins_centers < filtering_end_time_sec
)
filtering_data = filtering_data[valid_bins_mask, :]
filtering_bins_centers = bins_centers[filtering_valid_bins_mask]

filtering_res = ssm.inference.filterLDS_SS_withMissingValues_np(
    y=filtering_data.T, u=optim_res["u"], B=optim_res["B"], Q=optim_res["Q"],
    m0=optim_res["m0"], V0=optim_res["V0"], a=optim_res["a"], Z=optim_res["Z"],
    R=optim_res["R"])

## Plot filtered states



In [16]:
# define auxiliary function
def build_events_df(start_time_sec, end_time_sec, transitions_data,
                    ports_linetypes, ports_colors,
                    start_poke_in_time_col_name="P1_IN_Ephys_TS",
                    start_port_col_name="Start_Port"):
    mask = np.logical_and(
        start_time_sec<=transitions_data[start_poke_in_time_col_name],
        transitions_data[start_poke_in_time_col_name]<end_time_sec
    )
    subset_transitions_data = transitions_data[mask]
    event_time = subset_transitions_data[start_poke_in_time_col_name]
    ports_names = subset_transitions_data[start_port_col_name]
    event_line_type = [ports_linetypes[port_name]
                       for port_name in ports_names]
    event_color = [ports_colors[port_name]
                   for port_name in ports_names]
    answer = pd.DataFrame(dict(event_time=event_time,
                               event_line_type=event_line_type,
                               event_color=event_color))
    return answer


transitions_data = pd.read_csv(transition_data_filename)

events_df = build_events_df(
    start_time_sec=filtering_start_time_sec,
    end_time_sec=filtering_end_time_sec,
    transitions_data=transitions_data,
    ports_linetypes=ports_linetypes,
    ports_colors=ports_colors)

plotting_end_time_sec = plotting_start_time_sec + plotting_duration_sec
plotting_first_index = np.where(filtering_bins_centers >= plotting_start_time_sec)[0][0]
plotting_last_index = np.where(filtering_bins_centers <= plotting_end_time_sec)[0][-1]
plotting_slice = slice(plotting_first_index, plotting_last_index)
plotting_bins_centers = filtering_bins_centers[plotting_slice]
plotting_means = filtering_res["xnn"][:,:,plotting_slice]
plotting_covs = filtering_res["Pnn"][:,:,plotting_slice]

fig = ssm.neural_latents.plotting.getPlotLatents(
    means=plotting_means,
    covs=plotting_covs,
    bins_centers=plotting_bins_centers,
    orthonormalize=not skip_orthonormalization,
    Z=optim_res["Z"],
)
ssm.plotting.add_events_vlines(fig=fig, events_df=events_df)

nro_samples = len(bins_centers)
title = ('Log-Likelihood per sample: ' f'{filtering_res["log_like"].squeeze() / nro_samples:.2E}')
fig.update_layout(title=title)
fig